# 03 · Camada Gold (CineData Analytics)

Modelagem dimensional (Star Schema) para consumo do time de BI.

## 1. Tabelas Dimensionais (Dimensões de Negócio)

**Objetivos:**
- Criação das chaves substitutas (Surrogate Keys - SK) utilizando `row_number()`, convertidas para `BIGINT` (o `row_number()` nativo é `INT`).
- Geração da `dim_movies` a partir dos metadados dos filmes.
- Geração da `dim_genres` deduplicada.
- Separação da tabela unificada da Silver em duas dimensões distintas: `dim_people` (Físicas) e `dim_companies` (Jurídicas).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "cinedata_analytics"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {gold_schema}")
spark.sql(f"USE {gold_schema}")

print(f"[OK] Base de dados {gold_schema} criada e definida como ativa.")

[OK] Base de dados cinedata_analytics.gold criada e definida como ativa.


In [0]:
def gravar_gold(df, nome_tabela):
    """Grava em Delta (overwrite) na Gold. Reexecutar o notebook recria as tabelas de forma idempotente."""
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{gold_schema}.{nome_tabela}")

# --- gold.dim_movies ---
df_silver_info = spark.table(f"{silver_schema}.tb_info_filmes")
window_movies = Window.orderBy("id_filme")

df_dim_movies = (
    df_silver_info
    .withColumn("sk_movie_id", F.row_number().over(window_movies).cast("bigint"))
    .select(
        "sk_movie_id",
        F.col("id_filme").cast("string").alias("id_filme"),
        "titulo",
        F.col("data_lancamento").cast("date").alias("data_lancamento"),
        F.col("ano_lancamento").cast("int").alias("ano_lancamento"),
        F.col("duracao_minutos").cast("int").alias("duracao_minutos"),
        "idioma_original", "status_filme", "sinopse"
    )
)
gravar_gold(df_dim_movies, "dim_movies")

# --- gold.dim_genres ---
df_silver_generos = spark.table(f"{silver_schema}.tb_generos")
window_genres = Window.orderBy("nome_genero")

df_dim_genres = (
    df_silver_generos
    .select("nome_genero").dropDuplicates()
    .withColumn("sk_genre_id", F.row_number().over(window_genres).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)
gravar_gold(df_dim_genres, "dim_genres")

# --- gold.dim_people (pessoas físicas) ---
df_silver_pessoas_empresas = spark.table(f"{silver_schema}.tb_pessoas_empresas")
window_people = Window.orderBy("nome_pessoa", "tipo_pessoa")

df_dim_people = (
    df_silver_pessoas_empresas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select("nome_pessoa", F.col("tipo_entidade").alias("tipo_pessoa"))
    .dropDuplicates()
    .withColumn("sk_person_id", F.row_number().over(window_people).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)
gravar_gold(df_dim_people, "dim_people")

# --- gold.dim_companies (pessoas jurídicas) ---
window_companies = Window.orderBy("nome_produtora")

df_dim_companies = (
    df_silver_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_pessoa").alias("nome_produtora"))
    .dropDuplicates()
    .withColumn("sk_company_id", F.row_number().over(window_companies).cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)
gravar_gold(df_dim_companies, "dim_companies")

print("[OK] dim_movies, dim_genres, dim_people e dim_companies processadas.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] dim_movies, dim_genres, dim_people e dim_companies processadas.


## 2. Fato, Avaliações e Tabelas-Ponte

**Objetivos:**
- Criação da `dim_reviews` com agregação por filme (contagem e média das notas).
- Construção da `fact_movies_performance` (grão: um registro por filme **lançado**) unindo financeiro e engajamento da Silver via `sk_movie_id`.
- Criação das tabelas-ponte (Bridges), cruzando as chaves naturais para obter as Surrogate Keys de cada dimensão.

In [0]:
# --- gold.dim_reviews ---
df_silver_avaliacoes = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

df_reviews_agg = (
    df_silver_avaliacoes
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner")
    .groupBy("sk_movie_id")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    )
)
df_dim_reviews = (
    df_reviews_agg
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")).cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)
gravar_gold(df_dim_reviews, "dim_reviews")


# --- gold.fact_movies_performance ---
# Silver financeiro e engajamento têm 1 linha por filme, então os LEFT JOINs não duplicam o grão.
df_silver_fin = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_silver_eng = spark.table(f"{silver_schema}.tb_metricas_engajamento")

df_fact = (
    df_dim_movies
    .filter(F.col("status_filme") == "Lançado")      # só filmes lançados (requisito do PDF)
    .select("sk_movie_id", "id_filme")
    .join(df_silver_fin, "id_filme", "left")
    .join(df_silver_eng, "id_filme", "left")
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int")
    )
)
gravar_gold(df_fact, "fact_movies_performance")


# --- Tabelas-Ponte (Bridges) ---
df_dim_genres = spark.table(f"{gold_schema}.dim_genres")
df_dim_people = spark.table(f"{gold_schema}.dim_people")
df_dim_companies = spark.table(f"{gold_schema}.dim_companies")
df_dim_movies_keys = df_dim_movies.select("sk_movie_id", "id_filme")

# bridge_movie_genre
df_bridge_genre = (
    spark.table(f"{silver_schema}.tb_generos")
    .join(df_dim_movies_keys, "id_filme", "inner")
    .join(df_dim_genres, "nome_genero", "inner")
    .select("sk_movie_id", "sk_genre_id").dropDuplicates()
)
gravar_gold(df_bridge_genre, "bridge_movie_genre")

# bridge_movie_person (a chave da pessoa é nome + tipo, igual à dim_people)
df_pessoas_fisicas = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .withColumnRenamed("tipo_entidade", "tipo_pessoa")
)
df_bridge_person = (
    df_pessoas_fisicas
    .join(df_dim_movies_keys, "id_filme", "inner")
    .join(df_dim_people, ["nome_pessoa", "tipo_pessoa"], "inner")
    .select("sk_movie_id", "sk_person_id").dropDuplicates()
)
gravar_gold(df_bridge_person, "bridge_movie_person")

# bridge_movie_company
df_produtoras = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .withColumnRenamed("nome_pessoa", "nome_produtora")
)
df_bridge_company = (
    df_produtoras
    .join(df_dim_movies_keys, "id_filme", "inner")
    .join(df_dim_companies, "nome_produtora", "inner")
    .select("sk_movie_id", "sk_company_id").dropDuplicates()
)
gravar_gold(df_bridge_company, "bridge_movie_company")

print("[OK] Fato, dim_reviews e Bridges processadas com sucesso.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] Fato, dim_reviews e Bridges processadas com sucesso.
